In [2]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.regression.mixed_linear_model import MixedLM

In [50]:
combined_df = pd.read_excel('Combined_Time_Fix_Data.xlsx')
Dataset_Values = combined_df['Dataset'].unique()
columns = combined_df.columns.values
combined_df.drop(columns=columns[0], axis=1, inplace=True) # inplace to modify combined_df instead of copy of object
combined_df
combined_df[combined_df['Dataset']==Dataset_Values[2]].head()

,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS
600,SOS,19232,SOS_Spontaneous,9.864331,True,True,True,True
601,SOS,21931,SOS_Spontaneous,9.995656,True,True,True,True
602,SOS,19650,SOS_Spontaneous,9.885833,True,True,True,True
603,SOS,26696,SOS_Spontaneous,10.192269,True,True,True,True
604,SOS,17714,SOS_Spontaneous,9.782111,True,True,True,True


### Conjugation R_a Regression Analysis

In [39]:
# Filter the dataframe to exclude the 'Chemotaxis_Diffusion' dataset
filtered_df = combined_df[combined_df['Dataset'] == 'Conjugation_Ra'].copy()

# Add interaction term for R_a * Conjugation
filtered_df['R_a_Conj'] = filtered_df['R_a'] & filtered_df['Conjugation']

# Display the first few rows to verify the new column
filtered_df[['Condition', 'Time', 'Dataset', 'Log_Time', 'Conjugation', 'R_a', 'R_a_Conj']].head()

,Condition,Time,Dataset,Log_Time,Conjugation,R_a,R_a_Conj
200,Normal Equation,1640,Conjugation_Ra,7.402452,True,True,True
201,Normal Equation,6705,Conjugation_Ra,8.810609,True,True,True
202,Normal Equation,4174,Conjugation_Ra,8.336630,True,True,True
203,Normal Equation,5143,Conjugation_Ra,8.545392,True,True,True
204,Normal Equation,6589,Conjugation_Ra,8.793157,True,True,True


In [40]:
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ Conjugation + R_a'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.014
Method:                 Least Squares   F-statistic:                     3.873
Date:                Wed, 03 Jul 2024   Prob (F-statistic):             0.0216
Time:                        16:34:03   Log-Likelihood:                -311.03
No. Observations:                 400   AIC:                             628.1
Df Residuals:                     397   BIC:                             640.0
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                          coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------
Intercept               8.7860    

In [41]:
filtered_df['R_a_Conj'].info()
# Prepare the model formula including the interaction term
formula = 'Log_Time ~ R_a_Conj'

# Fit the OLS regression model
model_interaction = sm.OLS.from_formula(formula, data=filtered_df).fit()

# Display the summary of the model
model_interaction_summary = model_interaction.summary()
print(model_interaction_summary)

<class 'pandas.core.series.Series'>
Index: 400 entries, 200 to 599
Series name: R_a_Conj
Non-Null Count  Dtype
--------------  -----
400 non-null    bool 
dtypes: bool(1)
memory usage: 3.5 KB
                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.018
Model:                            OLS   Adj. R-squared:                  0.015
Method:                 Least Squares   F-statistic:                     7.121
Date:                Wed, 03 Jul 2024   Prob (F-statistic):            0.00793
Time:                        16:34:07   Log-Likelihood:                -311.35
No. Observations:                 400   AIC:                             626.7
Df Residuals:                     398   BIC:                             634.7
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  

The interaction term R_a_Conj is statistically significant in the simpler model because this model directly captures the combined effect without the added degrees of freedom, larger standard errors, and potential collinearity of separate terms, thereby reducing the t-values and making it harder to achieve statistical significance.

Conclusion of Analysis and Comparison to Permutation Test:
- Conjugation is not a statistically significant predictor. (Coefficient is -0.0435, not significant p = 0.412).
- R_a is significant. (Coefficient is -0.1405, significant p = 0.008).
- R_a*Conj, Analysis with Interaction Term Only: (Coefficient is -0.1628, significant p = 0.008).
- Significant in the model focusing on interaction, indicating a combined effect that is crucial.
- The permutation test showed practical significance for the interaction effect but did not reach statistical significance.
- Regression results for R_a_Conj indicate a significant interaction effect in the model focusing on this term, aligning with the practical importance suggested by the permutation test.
- A higher R-squared indicates more variance in log('Time'), and higher F-statistic indicates that the model is more statistically significant, both indicate the model is a better fit.

### Chemotaxis Diffusion Regression Analysis

In [67]:
# Filter the dataframe to include only the 'Chemotaxis_Diffusion' dataset
chemotaxis_df = combined_df[combined_df['Dataset'] == 'Chemotaxis_Diffusion'].copy()

# Display the first few rows to verify
chemotaxis_df[['Condition', 'Time', 'Dataset', 'Log_Time', 'Chemotaxis']].head()

,Condition,Time,Dataset,Log_Time,Chemotaxis
0,Chemotaxis,19232,Chemotaxis_Diffusion,9.864331,1.0
1,Chemotaxis,11287,Chemotaxis_Diffusion,9.331407,1.0
2,Chemotaxis,21931,Chemotaxis_Diffusion,9.995656,1.0
3,Chemotaxis,19650,Chemotaxis_Diffusion,9.885833,1.0
4,Chemotaxis,26696,Chemotaxis_Diffusion,10.192269,1.0


In [68]:

# Prepare the model formula
formula = 'Log_Time ~ Chemotaxis'

# Fit the OLS regression model
model_chemotaxis = sm.OLS.from_formula(formula, data=chemotaxis_df).fit()

# Display the summary of the model
model_chemotaxis_summary = model_chemotaxis.summary()
print(model_chemotaxis_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.085
Model:                            OLS   Adj. R-squared:                  0.080
Method:                 Least Squares   F-statistic:                     18.29
Date:                Tue, 02 Jul 2024   Prob (F-statistic):           2.95e-05
Time:                        17:55:31   Log-Likelihood:                -10.991
No. Observations:                 200   AIC:                             25.98
Df Residuals:                     198   BIC:                             32.58
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     10.1139      0.026    393.645      0.0

### Mixed-Effects Model

In [56]:
# Step 1: Separate Regression Models

# Filter data for Chemotaxis_Diffusion
chemotaxis_df = combined_df[combined_df['Dataset'] == 'Chemotaxis_Diffusion'].copy()
# Fit OLS regression model for Chemotaxis
model_chemotaxis = sm.OLS.from_formula('Log_Time ~ Chemotaxis', data=chemotaxis_df).fit()
model_chemotaxis_summary = model_chemotaxis.summary()
print(model_chemotaxis_summary)

# Filter data for Conjugation_Ra
conjugation_ra_df = combined_df[combined_df['Dataset'] == 'Conjugation_Ra'].copy()
# Add interaction term for R_a * Conjugation
conjugation_ra_df['R_a_Conj'] = conjugation_ra_df['R_a'] & conjugation_ra_df['Conjugation']
# Fit OLS regression model for R_a_Conj
model_ra_conj = sm.OLS.from_formula('Log_Time ~ R_a_Conj', data=conjugation_ra_df).fit()
model_ra_conj_summary = model_ra_conj.summary()
print(model_ra_conj_summary)

# Filter data for SOS_Spontaneous
sos_spontaneous_df = combined_df[combined_df['Dataset'] == 'SOS_Spontaneous'].copy()
# Fit OLS regression model for R_a_Conj
model_sos_spontaneous = sm.OLS.from_formula('Log_Time ~ SOS', data=sos_spontaneous_df).fit()
model_sos_spontaneous_summary = model_sos_spontaneous.summary()
print(model_sos_spontaneous_summary)

                            OLS Regression Results                            
Dep. Variable:               Log_Time   R-squared:                       0.085
Model:                            OLS   Adj. R-squared:                  0.080
Method:                 Least Squares   F-statistic:                     18.29
Date:                Wed, 03 Jul 2024   Prob (F-statistic):           2.95e-05
Time:                        16:43:44   Log-Likelihood:                -10.991
No. Observations:                 200   AIC:                             25.98
Df Residuals:                     198   BIC:                             32.58
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
Intercept             10.1139      0

In [58]:
# Step 2: Mixed-Effects Model
# Combine both datasets for mixed-effects model
combined_df_mixed = pd.concat([chemotaxis_df, conjugation_ra_df, sos_spontaneous_df], ignore_index=True)
combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis']
# Set 'Chemotaxis' values to NaN for the 'Conjugation_Ra' dataset
combined_df_mixed['Chemotaxis'] = combined_df_mixed['Chemotaxis'].astype(bool)
# combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis'] = pd.NA
combined_df_mixed['R_a_Conj'] = combined_df_mixed['R_a_Conj'].astype(bool)
# combined_df_mixed.loc[combined_df_mixed['Dataset'] == 'Conjugation_Ra', 'Chemotaxis'] = pd.NA


# Verify the changes
combined_df_mixed.info()
combined_df_mixed.tail()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 794 entries, 0 to 793
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Condition    794 non-null    object 
 1   Time         794 non-null    int64  
 2   Dataset      794 non-null    object 
 3   Log_Time     794 non-null    float64
 4   Conjugation  794 non-null    bool   
 5   R_a          794 non-null    bool   
 6   Chemotaxis   794 non-null    bool   
 7   SOS          794 non-null    bool   
 8   R_a_Conj     794 non-null    bool   
dtypes: bool(5), float64(1), int64(1), object(2)
memory usage: 28.8+ KB


,Condition,Time,Dataset,Log_Time,Conjugation,R_a,Chemotaxis,SOS,R_a_Conj
789,Spontaneous,61984,SOS_Spontaneous,11.034637,True,True,True,False,True
790,Spontaneous,61984,SOS_Spontaneous,11.034637,True,True,True,False,True
791,Spontaneous,46438,SOS_Spontaneous,10.745873,True,True,True,False,True
792,Spontaneous,61984,SOS_Spontaneous,11.034637,True,True,True,False,True
793,Spontaneous,48751,SOS_Spontaneous,10.794481,True,True,True,False,True


In [62]:
# Prepare the mixed-effects model formula
mixed_formula = 'Log_Time ~ Chemotaxis + R_a_Conj + SOS'

# Fit the mixed-effects model
mixed_model = MixedLM.from_formula(mixed_formula, groups='Dataset', data=combined_df_mixed, missing='drop').fit()

# Display the summary of the mixed-effects model
mixed_model_summary = mixed_model.summary()
print(mixed_model_summary)

            Mixed Linear Model Regression Results
Model:               MixedLM   Dependent Variable:   Log_Time 
No. Observations:    794       Method:               REML     
No. Groups:          3         Scale:                0.1754   
Min. group size:     194       Log-Likelihood:       -449.5936
Max. group size:     400       Converged:            Yes      
Mean group size:     264.7                                    
--------------------------------------------------------------
                   Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------
Intercept          10.650    0.470  22.679 0.000  9.730 11.571
Chemotaxis[T.True] -0.157    0.059  -2.645 0.008 -0.273 -0.041
R_a_Conj[T.True]   -0.160    0.048  -3.312 0.001 -0.255 -0.065
SOS[T.True]        -0.832    0.060 -13.836 0.000 -0.950 -0.714
Dataset Var         0.642    1.531                            

